In [4]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Read the CSV file
df = pd.read_csv('2024-25_cepg_recipients_web_en.csv')

# Clean the funding column
df['Funding $'] = df['Funding $'].astype(str).str.replace(',', '').astype(float)

# Create a function to parse and categorize expenditures based on the "Description" column
def categorize_recipient(Recipient):
    recipient_lower = Recipient.lower()
    if 'first nation' in recipient_lower or 'band council' in recipient_lower or 'mohawk' in recipient_lower:
        return 'First Nations'
    elif 'corporation of the city' in recipient_lower or 'city of' in recipient_lower:
        return 'Cities'
    elif 'corporation of the county' in recipient_lower or 'county of' in recipient_lower:
        return 'Counties'
    elif 'corporation of the town' in recipient_lower or 'town of' in recipient_lower:
        return 'Towns'
    elif 'corporation of the township' in recipient_lower or 'township of' in recipient_lower or 'corporation of the municipality' in recipient_lower or 'municipality of' in recipient_lower:
        return 'Townships/Municipalities'
    elif 'local services board' in recipient_lower:
        return 'Local Services Boards'
    elif 'search and rescue' in recipient_lower or 'fire protection' in recipient_lower:
        return 'Emergency Services'
    else:
        return 'Other Organizations'
    
# Create a function to categorize recipients
def categorize_expenditure(Description):
    description_lower = Description.lower()
    if 'generator' in description_lower:
        return 'Generators & Power'
    elif 'drone' in description_lower:
        return 'Drones & Surveillance'
    elif 'radio' in description_lower or 'communication' in description_lower:
        return 'Communications Equipment'
    elif 'flood' in description_lower or 'sandbag' in description_lower or 'pump' in description_lower:
        return 'Flood Response'
    elif 'fire' in description_lower or 'wildland' in description_lower or 'chainsaw' in description_lower:
        return 'Fire Response'
    elif 'training' in description_lower or 'education' in description_lower:
        return 'Training & Education'
    elif 'shelter' in description_lower or 'evacuation' in description_lower or 'emergency centre' in description_lower:
        return 'Emergency Shelters'
    elif 'rescue' in description_lower:
        return 'Search & Rescue'
    else:
        return 'Other Equipment'
    
# Apply categorization functions defined above
df['Recipient Type'] = df['Recipient'].apply(categorize_recipient)
df['Expenditure Category'] = df['Description'].apply(categorize_expenditure)

# PREPARE DATA FOR TABLEAU SANKEY EXTENSION
sankey_data = df.groupby(['Recipient Type', 'Expenditure Category'])['Funding $'].sum().reset_index()
sankey_data.columns = ['Source', 'Target', 'Value']

# Add some additional useful columns
sankey_data['Count'] = df.groupby(['Recipient Type', 'Expenditure Category']).size().reset_index(drop=True)
sankey_data['Avg_Funding'] = sankey_data['Value'] / sankey_data['Count']

# Sort by value for better visualization
sankey_data = sankey_data.sort_values('Value', ascending=False)

# Save for Tableau
sankey_data.to_csv('sankey_data_for_tableau.csv', index=False)